# Mission Eagle-1 — Optimisation du pilote automatique

Notre premier modele PPO (notebook 01) donne un score de **0.3** (cf. notebook 01). C'est mieux qu'un agent aleatoire, mais notre objectif est un score stable au-dessus de **200** — le seuil d'un atterrissage reussi.

**Strategie** : on va modifier les hyperparametres **un par un** pour isoler l'effet de chacun. C'est la methode scientifique appliquee au ML.


In [1]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
import time

## 1. Les hyperparametres de PPO

Avant de toucher a quoi que ce soit, comprenons ce que chaque parametre fait.

| Hyperparametre  | Defaut SB3 | Role                                                         | Analogie mission                                                                                               |
| --------------- | ---------- | ------------------------------------------------------------ | -------------------------------------------------------------------------------------------------------------- |
| `learning_rate` | 3e-4       | A quelle vitesse le pilote integre les nouvelles experiences | Reactivite du pilote — trop haut = instable, trop bas = apprend trop lentement                                 |
| `gamma`         | 0.99       | Poids des recompenses futures vs immediates                  | Horizon de planification : gamma proche de 1 = le pilote anticipe loin (economise du carburant pour plus tard) |
| `n_steps`       | 2048       | Nombre de steps collectes avant chaque mise a jour           | Taille du "carnet de vol" analyse entre chaque correction                                                      |
| `ent_coef`      | 0.0        | Bonus d'exploration — encourage l'agent a varier ses actions | Curiosite du pilote : ent_coef > 0 = il essaie des manoeuvres differentes                                      |
| `n_epochs`      | 10         | Nombre de passes sur les donnees collectees a chaque update  | Nombre de fois que le pilote "relit ses notes" avant de voler a nouveau                                        |

> **Note methodo** : On ne modifie **qu'un seul parametre a la fois**. Sinon, impossible de savoir lequel a fait la difference. C'est le principe de base de toute experimentation.


## 2. Experience 1 — Learning Rate

Le learning rate controle la vitesse d'apprentissage. On teste 3 valeurs :

- `1e-4` (prudent — apprend lentement mais surement)
- `3e-4` (defaut SB3)
- `1e-3` (agressif — apprend vite mais risque d'instabilite)

On entraine chaque modele sur **300 000 timesteps** — assez pour voir des tendances sans attendre trop longtemps.


In [2]:
learning_rates = [1e-4, 3e-4, 1e-3]
results_lr = []

for lr in learning_rates:
    print(f"\n{'=' * 55}")
    print(f"Entrainement PPO avec learning_rate={lr}")
    print(f"{'=' * 55}")

    env = gym.make("LunarLander-v3")
    model = PPO(
        "MlpPolicy", env, learning_rate=lr, verbose=0, tensorboard_log="./logs/exp1_lr"
    )

    start = time.time()
    model.learn(total_timesteps=300_000, tb_log_name=f"lr_{lr}")
    duration = time.time() - start

    eval_env = gym.make("LunarLander-v3")
    mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=50)
    eval_env.close()
    env.close()

    results_lr.append(
        {
            "lr": lr,
            "mean_reward": mean_reward,
            "std_reward": std_reward,
            "duration": duration,
        }
    )

    print(f"  Reward moyen : {mean_reward:.1f} +/- {std_reward:.1f}")
    print(f"  Temps : {duration:.0f}s")

print("\n\nRecap :")
print(f"{'LR':>10} | {'Reward':>10} | {'Std':>8} | {'Temps':>6}")
print("-" * 42)
for r in results_lr:
    print(
        f"{r['lr']:>10} | {r['mean_reward']:>10.1f} | {r['std_reward']:>8.1f} | {r['duration']:>5.0f}s"
    )


Entrainement PPO avec learning_rate=0.0001


/Users/ppluton/dev/tuto_baseline_rl/.venv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/Users/ppluton/dev/tuto_baseline_rl/.venv/lib/python3.11/site-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


  Reward moyen : -32.4 +/- 48.4
  Temps : 74s

Entrainement PPO avec learning_rate=0.0003
  Reward moyen : -73.6 +/- 36.8
  Temps : 71s

Entrainement PPO avec learning_rate=0.001
  Reward moyen : 184.9 +/- 58.1
  Temps : 83s


Recap :
        LR |     Reward |      Std |  Temps
------------------------------------------
    0.0001 |      -32.4 |     48.4 |    74s
    0.0003 |      -73.6 |     36.8 |    71s
     0.001 |      184.9 |     58.1 |    83s


### Analyse

Les resultats sont clairs :

- **lr=1e-4** (-32.4) : trop prudent, l'agent n'a pas du tout converge en 300k steps
- **lr=3e-4** (-73.6) : surprenant, le defaut SB3 echoue ici — l'apprentissage n'a pas decolle sur ce run
- **lr=1e-3** (184.9) : le LR agressif donne le **meilleur score**, tres proche du seuil de 200

On retient **lr=1e-3** pour la suite.

> **Note methodo** : Si un learning rate trop eleve fait diverger l'entrainement, ca se voit par un reward qui chute brutalement ou oscille beaucoup. Ici, 1e-3 fonctionne bien — ca depend de l'environnement. La variance entre runs en RL est connue : un meme hyperparametre peut donner des resultats tres differents selon le seed.


## 3. Experience 2 — Gamma (facteur d'actualisation)

Gamma controle a quel point l'agent se soucie du futur. Avec le meilleur learning rate trouve ci-dessus, on teste :

- `0.99` (defaut — bon equilibre)
- `0.995` (regarde un peu plus loin)
- `0.999` (tres long terme)

> **Rappel concept** : `gamma = 0.99` signifie qu'une recompense dans 100 steps vaut `0.99^100 = 0.37` fois sa valeur immediate. Avec `gamma = 0.999`, elle vaut `0.999^100 = 0.90`. L'agent planifie beaucoup plus loin.


In [3]:
best_lr = results_lr[np.argmax([r["mean_reward"] for r in results_lr])]["lr"]
print(f"Meilleur LR de l'exp 1 : {best_lr}")

gammas = [0.99, 0.995, 0.999]
results_gamma = []

for gamma in gammas:
    print(f"\n{'=' * 55}")
    print(f"PPO avec lr={best_lr}, gamma={gamma}")
    print(f"{'=' * 55}")

    env = gym.make("LunarLander-v3")
    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=best_lr,
        gamma=gamma,
        verbose=0,
        tensorboard_log="./logs/exp2_gamma",
    )

    start = time.time()
    model.learn(total_timesteps=300_000, tb_log_name=f"gamma_{gamma}")
    duration = time.time() - start

    eval_env = gym.make("LunarLander-v3")
    mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=50)
    eval_env.close()
    env.close()

    results_gamma.append(
        {
            "gamma": gamma,
            "mean_reward": mean_reward,
            "std_reward": std_reward,
            "duration": duration,
        }
    )

    print(f"  Reward moyen : {mean_reward:.1f} +/- {std_reward:.1f}")

print("\n\nRecap :")
print(f"{'Gamma':>10} | {'Reward':>10} | {'Std':>8}")
print("-" * 34)
for r in results_gamma:
    print(f"{r['gamma']:>10} | {r['mean_reward']:>10.1f} | {r['std_reward']:>8.1f}")

Meilleur LR de l'exp 1 : 0.001

PPO avec lr=0.001, gamma=0.99
  Reward moyen : 212.4 +/- 79.9

PPO avec lr=0.001, gamma=0.995
  Reward moyen : 249.8 +/- 39.2

PPO avec lr=0.001, gamma=0.999
  Reward moyen : 231.5 +/- 70.9


Recap :
     Gamma |     Reward |      Std
----------------------------------
      0.99 |      212.4 |     79.9
     0.995 |      249.8 |     39.2
     0.999 |      231.5 |     70.9


## 4. Experience 3 — n_steps (taille du batch)

`n_steps` determine combien de pas l'agent collecte avant de mettre a jour sa policy. C'est un compromis :

- **Petit n_steps** : mises a jour frequentes, mais estimations bruitees
- **Grand n_steps** : mises a jour plus rares, mais estimations plus stables

On teste avec les meilleurs LR et gamma trouves.


In [4]:
best_gamma = results_gamma[np.argmax([r["mean_reward"] for r in results_gamma])][
    "gamma"
]
print(f"Meilleurs params : lr={best_lr}, gamma={best_gamma}")

n_steps_list = [1024, 2048, 4096]
results_nsteps = []

for n_steps in n_steps_list:
    print(f"\n{'=' * 55}")
    print(f"PPO avec lr={best_lr}, gamma={best_gamma}, n_steps={n_steps}")
    print(f"{'=' * 55}")

    env = gym.make("LunarLander-v3")
    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=best_lr,
        gamma=best_gamma,
        n_steps=n_steps,
        verbose=0,
        tensorboard_log="./logs/exp3_nsteps",
    )

    start = time.time()
    model.learn(total_timesteps=300_000, tb_log_name=f"nsteps_{n_steps}")
    duration = time.time() - start

    eval_env = gym.make("LunarLander-v3")
    mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=50)
    eval_env.close()
    env.close()

    results_nsteps.append(
        {
            "n_steps": n_steps,
            "mean_reward": mean_reward,
            "std_reward": std_reward,
            "duration": duration,
        }
    )

    print(f"  Reward moyen : {mean_reward:.1f} +/- {std_reward:.1f}")

print("\n\nRecap :")
print(f"{'n_steps':>10} | {'Reward':>10} | {'Std':>8}")
print("-" * 34)
for r in results_nsteps:
    print(f"{r['n_steps']:>10} | {r['mean_reward']:>10.1f} | {r['std_reward']:>8.1f}")

Meilleurs params : lr=0.001, gamma=0.995

PPO avec lr=0.001, gamma=0.995, n_steps=1024
  Reward moyen : -77.5 +/- 166.5

PPO avec lr=0.001, gamma=0.995, n_steps=2048
  Reward moyen : 127.2 +/- 113.7

PPO avec lr=0.001, gamma=0.995, n_steps=4096
  Reward moyen : 267.9 +/- 43.0


Recap :
   n_steps |     Reward |      Std
----------------------------------
      1024 |      -77.5 |    166.5
      2048 |      127.2 |    113.7
      4096 |      267.9 |     43.0


## 5. Entrainement final — Le meilleur modele

On a nos meilleurs hyperparametres. Maintenant on entraine un modele "serieusement" — plus de timesteps pour lui laisser le temps de converger.

On passe a **500 000 timesteps** (voire 1M si le score n'est pas assez stable). Et on evalue sur **100 episodes** pour une mesure vraiment fiable.


In [5]:
best_nsteps = results_nsteps[np.argmax([r["mean_reward"] for r in results_nsteps])][
    "n_steps"
]

print("Meilleurs hyperparametres trouves :")
print(f"  learning_rate = {best_lr}")
print(f"  gamma         = {best_gamma}")
print(f"  n_steps       = {best_nsteps}")
print()

env = gym.make("LunarLander-v3")

model_final = PPO(
    "MlpPolicy",
    env,
    learning_rate=best_lr,
    gamma=best_gamma,
    n_steps=best_nsteps,
    verbose=1,
    tensorboard_log="./logs/final",
)

print("Entrainement final — 500k timesteps")
print("=" * 55)
model_final.learn(total_timesteps=500_000, tb_log_name="ppo_optimized")
env.close()

# Evaluation sur 100 episodes
eval_env = gym.make("LunarLander-v3")
mean_reward, std_reward = evaluate_policy(model_final, eval_env, n_eval_episodes=100)
eval_env.close()

print(f"\nResultat final :")
print(f"  Reward moyen : {mean_reward:.1f} +/- {std_reward:.1f}")
print(f"  Objectif 200+: {'OUI !' if mean_reward > 200 else 'Pas encore...'}")

# Sauvegarde
model_final.save("models/ppo_optimized")
print(f"\nModele sauvegarde dans models/ppo_optimized.zip")

Meilleurs hyperparametres trouves :
  learning_rate = 0.001
  gamma         = 0.995
  n_steps       = 4096

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Entrainement final — 500k timesteps
Logging to ./logs/final/ppo_optimized_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 90.3     |
|    ep_rew_mean     | -173     |
| time/              |          |
|    fps             | 7051     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 4096     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 91.7        |
|    ep_rew_mean          | -166        |
| time/                   |             |
|    fps                  | 5391        |
|    iterations           | 2           |
|    time_elapsed         | 1           |
|    total_timesteps      | 8192        |
| tr

## Recap de toutes les experiences

| Experience | Parametre     | Valeurs testees        | Meilleur                   |
| ---------- | ------------- | ---------------------- | -------------------------- |
| Exp 1      | learning_rate | 1e-4, 3e-4, 1e-3       | **1e-3** (184.9)           |
| Exp 2      | gamma         | 0.99, 0.995, 0.999     | **0.995** (249.8)          |
| Exp 3      | n_steps       | 1024, 2048, 4096       | **4096** (267.9)           |
| **Final**  | **Tous**      | **Meilleurs combines** | **Score : 242.6 +/- 58.2** |

L'objectif de 200 est atteint avec une marge confortable. L'ecart-type reste un peu eleve (58.2), ce qui signifie que certains episodes se passent moins bien que d'autres — mais en moyenne, notre pilote sait atterrir.

> **Note methodo** : Le tuning d'hyperparametres est un processus iteratif. On pourrait aller plus loin (tester d'autres valeurs, faire plusieurs seeds par config...) mais l'objectif est d'atteindre 200, pas de trouver le modele parfait. Savoir s'arreter, c'est aussi une competence.

**Prochaine etape** : comparer avec DQN ->
